<a href="https://colab.research.google.com/github/ab23ms233/GRISHMA-iitkgp-2026/blob/main/Mechanistic-interpretability/interpretability_intro_to_transformerlens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cloning GitHub repository

In [ ]:
!git clone https://github.com/callummcdougall/ARENA_3.0.git

# Setup

## Optimal Installation Flow to Prevent Import Errors

To prevent the `OSError` and `ImportError` issues related to `torch`, `torchaudio`, and `transformers` compatibility, it's essential to install these libraries in a specific order and, in some cases, pin their versions.

1.  **Install `jaxtyping`**: A general dependency.
2.  **Install `transformer_lens`**: This is a key step, as it will install `torch` (often downgrading it to `2.7.1` or a similar version).
3.  **Uninstall `torchaudio` and `transformers`**: This ensures that any previously installed, incompatible versions are removed.
4.  **Install `torchaudio` with a pinned version**: We explicitly install `torchaudio==2.7.1` to match the `torch` version installed by `transformer_lens`.
5.  **Install `transformers`**: With `torch` and `torchaudio` now aligned, `pip` will be able to find a compatible `transformers` version.
6.  **Install `circuitsvis`**: Another general dependency.

In [ ]:
# Optimal installation commands

# 1. Install jaxtyping
!pip install jaxtyping

# 2. Install transformer_lens (this will set the torch version, likely to 2.7.1)
!pip install transformer_lens

# 3. Uninstall torchaudio and transformers to clear any incompatible versions
!pip uninstall -y torchaudio transformers

# 4. Install torchaudio, explicitly pinning it to the torch version (e.g., 2.7.1)
!pip install torchaudio==2.7.1

# 5. Install transformers (this should now pick a version compatible with torchaudio==2.7.1)
!pip install transformers

# 6. Install circuitsvis
!pip install circuitsvis

In [ ]:
!pip install eindex

## Changing directory

In [ ]:
%cd /content/ARENA_3.0/chapter1_transformer_interp/exercises/

## Importing libraries and exercises

In [ ]:
import functools
import sys
from pathlib import Path
from typing import Callable

import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
from eindex.indexing import eindex
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utilities,
)
from transformer_lens.hook_points import HookPoint

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part2_intro_to_mech_interp"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_intro_to_mech_interp.tests as tests
from plotly_utils import (
    hist,
    imshow,
    plot_comp_scores,
    plot_logit_attribution,
    plot_loss_difference,
)

# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

# PART 1: TransformerLens Inroduction

## Importing GPT-2 small

In [ ]:
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

## Examining architecture

In [ ]:
gpt2_small.cfg

## Calculating loss

We calculate the loss of the model in predicting the next token in the sequence.

In [ ]:
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model out, let's find the loss on this paragraph!"""

loss = gpt2_small(model_description_text, return_type="loss")
print("Model loss:", loss)


## Tokenization

We explore the different methods for tokenization of a text

In [ ]:
print(gpt2_small.to_str_tokens("gpt2"))
print(gpt2_small.to_str_tokens(["gpt2", "gpt2"]))
print(gpt2_small.to_tokens("gpt2"))
print(gpt2_small.to_string([50256, 70, 457, 17]))

## Checking correctly predicted tokens

We calculate what and how many tokens are correctly predicted

In [ ]:
logits: Tensor = gpt2_small(model_description_text, return_type="logits")
prediction = logits.argmax(dim=-1).squeeze()[:-1]

# YOUR CODE HERE - get the model's prediction on the text
# Predictions are after the first token, hence omitting the 1st one
original = gpt2_small.to_tokens(model_description_text).squeeze()[1:]
correct = 0
correct_tokens = []

# Comparing the number of correct tokens
for tokid, origid in zip(prediction, original):
    if tokid == origid:
        token = gpt2_small.to_single_str_token(tokid.item())
        correct_tokens.append(token)
        correct += 1

print(f"Number of correct predictions: {correct}/{len(original)+1}")
print(f"Correctly predicted tokens: {correct_tokens}")

In [ ]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_tokens, remove_batch_dim=True)

print(type(gpt2_logits), type(gpt2_cache))
print(f"Number of tokens: {len(gpt2_tokens.squeeze())}")

## Calculating attention patterns

We calculate the attention patterns generated by the model and compare them with those that we get by from the query, key, and value matrices

In [ ]:
attn_patterns_from_shorthand = gpt2_cache["pattern", 0]
attn_patterns_from_full_name = gpt2_cache["blocks.0.attn.hook_pattern"]

t.testing.assert_close(attn_patterns_from_shorthand, attn_patterns_from_full_name)

In [ ]:
layer0_pattern_from_cache = gpt2_cache["pattern", 0]

# YOUR CODE HERE - define `layer0_pattern_from_q_and_k` manually, by manually performing the steps of the attention calculation (dot product, masking, scaling, softmax)

# HINT
# You'll need to use three different cache indexes in all:
# gpt2_cache["pattern", 0] to get the attention patterns, which have shape [nhead, seqQ, seqK]
# gpt2_cache["q", 0] to get the query vectors, which have shape [seqQ, nhead, headsize]
# gpt2_cache["k", 0] to get the key vectors, which have shape [seqK, nhead, headsize]

qvec, kvec = gpt2_cache["q", 0], gpt2_cache["k", 0]
headsize = qvec.shape[-1]
seqQ, seqK = qvec.shape[0], kvec.shape[0]

mask = t.tril(t.ones((seqQ, seqK), dtype=t.bool)).to(device)
attn_scores = einops.einsum(qvec, kvec, "seqQ nhead headsize, seqK nhead headsize -> nhead seqQ seqK")/t.sqrt(t.tensor(headsize))

inf = t.tensor(float("inf"))
layer0_pattern_from_q_and_k = t.where(mask, attn_scores, -inf).softmax(dim=-1)

t.testing.assert_close(layer0_pattern_from_cache, layer0_pattern_from_q_and_k)
print("Tests passed!")

In [ ]:
print(qvec.shape, kvec.shape)

## Visualizing attention patterns

We visualize the attention pattern of all the heads of the first layer

In [ ]:
print(type(gpt2_cache))
attention_pattern = gpt2_cache["pattern", 0]
print(attention_pattern.shape)
gpt2_str_tokens = gpt2_small.to_str_tokens(gpt2_text)

print("Layer 0 Head Attention Patterns:")
display(
    cv.attention.attention_patterns(
        tokens=gpt2_str_tokens,
        attention=attention_pattern
    )
)


## Visualizing Neuron Activations

We visualize the tokens that a neuron is activated to in different layers

In [ ]:
# Post means the activations after the GELU funtion of the MLP layer. We stack the activations of all the layers together.
neuron_activations_for_all_layers = t.stack([
    gpt2_cache["post", layer] for layer in range(gpt2_small.cfg.n_layers)
], dim=1)
# shape = (seq_pos, layers, neurons)

cv.activations.text_neuron_activations(
    tokens=gpt2_str_tokens,
    activations=neuron_activations_for_all_layers
)

In [ ]:
neuron_activations_for_all_layers_rearranged = utils.to_numpy(einops.rearrange(neuron_activations_for_all_layers, "seq layers neurons -> 1 layers seq neurons"))

cv.topk_tokens.topk_tokens(0,
    # Some weird indexing required here ¯\_(ツ)_/¯
    tokens=[gpt2_str_tokens],
    activations=neuron_activations_for_all_layers_rearranged,
    max_k=7,
    first_dimension_name="Layer",
    third_dimension_name="Neuron",
    first_dimension_labels=list(range(12))
)

# PART 2: Finding Induction Heads

## Configuration of toy model

The toy model has only two layers

In [ ]:
cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,  # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer",
)


## Importing toy model from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [ ]:
model = HookedTransformer(cfg)
pretrained_weights = t.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

## Visualizing attention

In [ ]:
text = "My name is ChatGPT. I am a trsnformer-based Large Language Model. I can do extraordinary stuffs. Try me out and tell me how you feel."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

# YOUR CODE HERE - visualize attention
tokens = model.to_str_tokens(text)

for i in range(model.cfg.n_layers):
    layer = cache["pattern", i]
    display(
    cv.attention.attention_patterns(
        tokens=tokens,
        attention=layer
    )
)

## Detecting attention head types

We detect the current token attention head, previous token attention head, and first token attention head

In [ ]:
def current_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
    """
    # List to store the current attention heads
    curr_heads = []
    false_threshold = 0.1

    # Iterating through the layers and heads
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            # Attention pattern of current layer and head
            attn_ptn = cache["pattern", layer][head]
            false_count = 0     # Variable to track number of diagonal elements which are not the greatest in their row

            # Checking if the diagonal element is greater than the max element in its row
            for row in range(attn_ptn.shape[0]):
                diag = attn_ptn[row, row]
                max_row = max(attn_ptn[row])

                if diag < max_row:
                    false_count += 1

            # Append head if fraction of false counts is less than threshold
            if false_count/attn_ptn.shape[0] < false_threshold:
                curr_heads.append(f"{layer}.{head}")

    return curr_heads


def prev_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
    """
    # List to store previous attention heads
    prev_heads = []
    false_threshold = 0.1

    # Iterating through the layers and heads
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            # Attention pattern of current layer and head
            attn_ptn = cache["pattern", layer][head]
            false_count = 0     # Variable to track number of diagonal elements which are not the greatest in their row

            # Checking if the off-diagonal element is greater than the max element in its row
            for row in range(1, attn_ptn.shape[0]):
                prev_diag = attn_ptn[row, row-1]
                max_row = max(attn_ptn[row])

                if prev_diag < max_row:
                    false_count += 1

            # Append head if fraction of false counts is less than threshold
            if false_count/attn_ptn.shape[0] < false_threshold:
                prev_heads.append(f"{layer}.{head}")

    return prev_heads


def first_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    """
    # List to store first attention heads
    first_heads = []
    false_threshold = 0.1

    # Iterating through the layers and heads
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            # Attention pattern of current layer and head
            attn_ptn = cache["pattern", layer][head]
            false_count = 0     # Variable to track number of diagonal elements which are not the greatest in their row

            # Checking if the 1st element is greater than the max element in its row
            for row in range(1, attn_ptn.shape[0]):
                first = attn_ptn[row, 0]
                max_row = max(attn_ptn[row])

                if first < max_row:
                    false_count +=1

            # Append head if fraction of false counts is less than threshold
            if false_count/attn_ptn.shape[0] < false_threshold:
                first_heads.append(f"{layer}.{head}")

    return first_heads


print("Heads attending to current token  = ", ", ".join(current_attn_detector(cache)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(cache)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(cache)))


## Generating repeated tokens

We write a function to generate repeated token sequences, train the model on such sequences and calculate loss.

In [ ]:
def generate_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Generates a sequence of repeated random tokens

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
    """
    t.manual_seed(0)  # for reproducibility
    # Beginning of sentence token
    prefix = (t.ones(batch_size, 1) * model.tokenizer.bos_token_id).long()
    # Generating the first sequence of tokens
    tokens = t.randint(0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.long)
    # Concatenating the sequence to form repeated tokens
    rep_tokens = t.concat((prefix, tokens, tokens), dim=-1).to(device)
    return rep_tokens


def run_and_cache_model_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> tuple[Tensor, Tensor, ActivationCache]:
    """
    Generates a sequence of repeated random tokens, and runs the model on it, returning (tokens,
    logits, cache). This function should use the `generate_repeated_tokens` function above.

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
        rep_logits: [batch_size, 1+2*seq_len, d_vocab]
        rep_cache: The cache of the model run on rep_tokens
    """
    rep_tokens = generate_repeated_tokens(model, seq_len, batch_size)
    rep_logits, rep_cache = model.run_with_cache(rep_tokens)

    return (rep_tokens, rep_logits, rep_cache)


def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    logprobs = logits.log_softmax(dim=-1)
    # We want to get logprobs[b, s, tokens[b, s+1]], in eindex syntax this looks like:
    correct_logprobs = eindex(logprobs, tokens, "b s [b s+1]")
    return correct_logprobs


seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_cache) = run_and_cache_model_repeated_tokens(model, seq_len, batch_size)
rep_cache.remove_batch_dim()
rep_str = model.to_str_tokens(rep_tokens)
model.reset_hooks()
log_probs = get_log_probs(rep_logits, rep_tokens).squeeze()

print(f"Performance on the first half: {log_probs[:seq_len].mean():.3f}")
print(f"Performance on the second half: {log_probs[seq_len:].mean():.3f}")

plot_loss_difference(log_probs, rep_str, seq_len)


## Visualizing induction heads

These are the attention heads in the 2nd layer which retain memory of pairs of tokens which occured together earlier in the sequence. If the same token occurs again, it is able to predict the next token after it.

In [ ]:
for i in range(model.cfg.n_layers):
    layer = rep_cache["pattern", i]
    display(
    cv.attention.attention_patterns(
        tokens=rep_str,
        attention=layer
    )
)

## Induction heads

These identify the next token for tokens which have occured together earlier in the sequence. We write a function to identify induction heads.

In [ ]:
def induction_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be induction heads

    Remember - the tokens used to generate rep_cache are (bos_token, *rand_tokens, *rand_tokens)
    """
    induc_heads = []
    seq_len = (cache["pattern", 0][0].shape[0]-1)//2
    false_threshold = 0.2

    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attn_ptn = cache["pattern", layer][head]
            false_count = 0

            for row in range(seq_len+1, attn_ptn.shape[0]):
                max_row = max(attn_ptn[row])

                if attn_ptn[row, row-seq_len+1] < max_row:
                    false_count += 1

            if false_count/attn_ptn.shape[0] < false_threshold:
                induc_heads.append(f"{layer}.{head}")

    return induc_heads




print("Induction heads = ", ", ".join(induction_attn_detector(rep_cache)))


In [ ]:
cache["pattern", 0].shape